In [ ]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime, timedelta
import time
import csv
import asyncio
import nest_asyncio
import logging
import websocket as ws_client
import xarray as xr
import threading

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)

usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [ ]:
async def trading_logic():
    global last_processed_candle, resampled_data
    while trading_active:
        current_time = pd.Timestamp.now()
        
        for token, data in resampled_data.items():
            if not data.empty:
                last_candle_start = data.index[-1]
                
                # Use a fixed 15-second interval if resample_freq is None
                resample_freq = data.index.freq or pd.Timedelta(seconds=5)
                
                current_candle_end = last_candle_start + resample_freq
                
                # if (current_time >= current_candle_end and
                #     (token not in last_processed_candle or current_candle_end > last_processed_candle[token])):
                    
                #     # Only process if this is truly a new candle
                #     if token not in last_processed_candle or current_candle_end > last_processed_candle[token] + resample_freq / 2:
                #         last_processed_candle[token] = current_candle_end
                                               
                        
                #         process_token_data(token)



                # Process the data only after the current candle has closed
                if current_time >= current_candle_end:
                    if token not in last_processed_candle or current_time >= last_processed_candle[token] + resample_freq:
                        last_processed_candle[token] = current_candle_end
                        process_token_data(token)

        await asyncio.sleep(0.1)  # Check every 10ms

def process_token_data(token):
    df = resampled_data[token]    

    if len(df) > ema_length:
        # Extract the latest and previous EMA values
        EMA_short_latest = df['EMA_short'].iloc[-2]
        EMA_long_latest = df['EMA_long'].iloc[-2]
        ALMA_latest = df['ALMA'].iloc[-2]
        EMA_short_previous = df['EMA_short'].iloc[-3]
        EMA_long_previous = df['EMA_long'].iloc[-3]       

        timestamp = df.index[-2]
        
        current_position = current_positions.get(token)
        action_taken = False

        # First, check for exit signals
        if current_position == 'long' and (EMA_short_latest < EMA_long_latest):
            log_trade(timestamp, token, "LONG_EXIT", EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous)
            print(f"Long Exit Signal for {token} at {df.index[-1]} at {pd.Timestamp.now()}")
            print(EMA_short_latest, EMA_long_latest, EMA_short_previous, EMA_long_previous, ALMA_latest)
            current_positions[token] = None
            action_taken = True

        elif current_position == 'short' and (EMA_short_latest > EMA_long_latest):
            log_trade(timestamp, token, "SHORT_EXIT", EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous)
            print(f"Short Exit Signal for {token} at {df.index[-1]} at {pd.Timestamp.now()}")
            print(EMA_short_latest, EMA_long_latest, EMA_short_previous, EMA_long_previous, ALMA_latest)
            current_positions[token] = None
            action_taken = True

        # Then, check for entry signals if we've just exited a position or if we're not in a position
        if action_taken or current_positions.get(token) is None:
            if (EMA_short_latest > EMA_long_latest >= ALMA_latest) and (EMA_short_previous <= EMA_long_previous):
                log_trade(timestamp, token, "LONG_ENTRY", EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous)
                print(f"Long Entry Signal for {token} at {df.index[-1]} at {pd.Timestamp.now()}")
                print(EMA_short_latest, EMA_long_latest, EMA_short_previous, EMA_long_previous, ALMA_latest)
                current_positions[token] = 'long'

            elif (EMA_short_latest < EMA_long_latest <= ALMA_latest) and (EMA_short_previous >= EMA_long_previous):
                log_trade(timestamp, token, "SHORT_ENTRY", EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous)
                print(f"Short Entry Signal for {token} at {df.index[-1]} at {pd.Timestamp.now()}")
                print(EMA_short_latest, EMA_long_latest, EMA_short_previous, EMA_long_previous, ALMA_latest)
                current_positions[token] = 'short'

def log_trade(timestamp, token, action, EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous):
    with open("trade_log.csv", "a") as log_file:
         log_file.write(f"{timestamp},{token},{action},{EMA_short_latest},{EMA_long_latest},{ALMA_latest},{EMA_short_previous},{EMA_long_previous}\n")

# Function to log trades
# def log_trade(timestamp, token, action, price):
#     with open("trade_log.csv", "a") as log_file:
#         log_file.write(f"{timestamp},{token},{action},{price}\n")

# # Dictionary to keep track of entry instruments
# entry_instruments = {}

# def get_latest_price(entry_instrument):
#     entry_instrument_str = str(entry_instrument)
#     if entry_instrument_str in extra_feedJson:
#         latest_data = extra_feedJson[entry_instrument_str][-1]
#         latest_price = latest_data['ltp']
#         print(f"Latest price for {entry_instrument_str} is {latest_price}")  # Debug print
#         return latest_price
#     else:
#         print(f"{entry_instrument_str} not found in extra_feedJson")  # Debug print
#         return None

# def process_token_data(token):
#     df = resampled_data[token]    

#     if len(df) > ema_length:
#         # Extract the latest and previous EMA values
#         EMA_short_latest = df['EMA_short'].iloc[-2]
#         EMA_long_latest = df['EMA_long'].iloc[-2]
#         ALMA_latest = df['ALMA'].iloc[-2]
#         EMA_short_previous = df['EMA_short'].iloc[-3]
#         EMA_long_previous = df['EMA_long'].iloc[-3]       

#         timestamp = df.index[-1]
        
#         current_position = current_positions.get(token)
#         action_taken = False

#         # First, check for exit signals
#         if current_position == 'long' and (EMA_short_latest < EMA_long_latest):
#             entry_instrument = entry_instruments.get(token, state.ce_trading_token)
#             price = get_latest_price(entry_instrument)
#             if price is not None:
#                 log_trade(timestamp, token, "LONG_EXIT", price)
#                 print(f"Long Exit Signal for {token} at {timestamp} at {pd.Timestamp.now()}")
#                 current_positions[token] = None
#                 action_taken = True

#         elif current_position == 'short' and (EMA_short_latest > EMA_long_latest):
#             entry_instrument = entry_instruments.get(token, state.pe_trading_token)
#             price = get_latest_price(entry_instrument)
#             if price is not None:
#                 log_trade(timestamp, token, "SHORT_EXIT", price)
#                 print(f"Short Exit Signal for {token} at {timestamp} at {pd.Timestamp.now()}")
#                 current_positions[token] = None
#                 action_taken = True

#         # Then, check for entry signals if we've just exited a position or if we're not in a position
#         if action_taken or current_positions.get(token) is None:
#             if (EMA_short_latest > EMA_long_latest >= ALMA_latest) and (EMA_short_previous <= EMA_long_previous):
#                 print("long")
#                 entry_instrument = state.ce_trading_token
#                 entry_instruments[token] = entry_instrument
#                 price = get_latest_price(entry_instrument)
#                 if price is not None:
#                     log_trade(timestamp, token, "LONG_ENTRY", price)
#                     print(f"Long Entry Signal for {token} at {timestamp} at {pd.Timestamp.now()}")
#                     current_positions[token] = 'long'

#             elif (EMA_short_latest < EMA_long_latest <= ALMA_latest) and (EMA_short_previous >= EMA_long_previous):
#                 print("SHORT")
#                 entry_instrument = state.pe_trading_token
#                 entry_instruments[token] = entry_instrument
#                 price = get_latest_price(entry_instrument)
#                 if price is not None:
#                     log_trade(timestamp, token, "SHORT_ENTRY", price)
#                     print(f"Short Entry Signal for {token} at {timestamp} at {pd.Timestamp.now()}")
#                     current_positions[token] = 'short'


In [ ]:
print(state.pe_trading_symbol)
price1 = get_latest_price(state.pe_trading_token)
price1

In [4]:
import asyncio
import threading
from datetime import datetime
import pandas as pd
import nest_asyncio

ema_length = 20
# Constants (you may want to adjust these values)
last_processed_candle = {}
initial_sl_points = 10
move_sl_points = 5
profit_booking_points = 20
trading_active = True
feed_opened = False
feedJson = {}
extra_feedJson = {}
resample_frequency = "5s"
resampling_lock = threading.Lock()
last_resample_time = {}
resampled_data = {}
current_positions = {}

ini = '428870'  ####################################### intial tokens number

if 'resampled_data' not in globals():
    resampled_data = {}
if 'last_resample_time' not in globals():
    last_resample_time = {}

def event_handler_feed_update(tick_data):
    
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with resampling_lock:
                if token == ini:
                    if token not in feedJson:
                        feedJson[token] = []
                        last_resample_time[token] = timest

                    feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
                
                else:
                    if token not in extra_feedJson:
                        extra_feedJson[token] = []
                        last_resample_time[token] = timest

                    extra_feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
                    if len(extra_feedJson[token]) > 10:
                        extra_feedJson[token].pop(0)

    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    
    while True:
        if not feedJson:
            await asyncio.sleep(1)
            continue

        with resampling_lock:
            for token, ticks in feedJson.items():
                try:
                    if ticks:
                        # Create a DataFrame with the new ticks
                        df_new = pd.DataFrame(ticks)
                        df_new["tt"] = pd.to_datetime(df_new["tt"])
                        df_new.set_index("tt", inplace=True)

                        # Determine the current resample interval
                        current_resample_interval = df_new.index.max().floor(resample_frequency)

                        # Initialize or update the resampled DataFrame
                        if token not in resampled_data:
                            # Initialize the DataFrame with the first resample
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()
                            resampled_data[token] = df_resampled
                            last_resample_time[token] = df_resampled.index.max()
                        else:
                            # Resample the new ticks
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()

                            # Update the existing resampled DataFrame with new data
                            for idx, row in df_resampled.iterrows():
                                if idx in resampled_data[token].index:
                                    resampled_data[token].loc[idx, 'high'] = max(resampled_data[token].loc[idx, 'high'], row['high'])
                                    resampled_data[token].loc[idx, 'low'] = min(resampled_data[token].loc[idx, 'low'], row['low'])
                                    resampled_data[token].loc[idx, 'close'] = row['close']
                                else:
                                    resampled_data[token].loc[idx] = row

                            # Update the last resampled time
                            last_resample_time[token] = df_resampled.index.max()

    
                        resampled_data[token]['EMA_short'] = ema_numba(resampled_data[token]['close'].values, length=3)
                        resampled_data[token]['EMA_long'] = ema_numba(resampled_data[token]['close'].values, length=10)
                        resampled_data[token]['ALMA'] = alma_numba(resampled_data[token]['close'].values, length=20, offset=0.75, sigma=6)
                        
                    feedJson[token] = []

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")

        await asyncio.sleep(1)
        

def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming `api` is defined and starts the WebSocket connection
async def connect_and_subscribe():
    global feed_opened
    api.start_websocket(
        order_update_callback=event_handler_order_update,
        subscribe_callback=event_handler_feed_update,
        socket_open_callback=open_callback
    )
    while not feed_opened:
        await asyncio.sleep(1)  # Wait for initial connection
    api.subscribe(['MCX|428870'])   

async def main():
   
    resample_task = asyncio.create_task(resample_ticks())
    trading_task = asyncio.create_task(trading_logic())
    connect_task = asyncio.create_task(connect_and_subscribe())  
    update_symbols_task = asyncio.create_task(update_atm_symbols())
    
    await asyncio.gather(connect_task, resample_task, trading_task, update_symbols_task)

def set_trading_active(value):
    global trading_active
    trading_active = value
    #print(f"Trading active set to {trading_active}")

# Get the current event loop
loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()
    asyncio.create_task(main())
else:
    loop.run_until_complete(main())

Short Entry Signal for 428870 at 2024-07-23 17:24:35 at 2024-07-23 17:24:40.101173
6543.94 6544.49 6544.87 6544.82 6545.72
Short Exit Signal for 428870 at 2024-07-23 17:25:05 at 2024-07-23 17:25:10.027842
6544.97 6544.76 6543.94 6544.49 6545.78
Short Entry Signal for 428870 at 2024-07-23 17:25:35 at 2024-07-23 17:25:40.006527
6544.49 6544.66 6544.98 6544.81 6545.46
Short Exit Signal for 428870 at 2024-07-23 17:25:55 at 2024-07-23 17:26:00.054946
6545.25 6544.9 6544.49 6544.66 6545.3
Short Entry Signal for 428870 at 2024-07-23 17:31:10 at 2024-07-23 17:31:15.104344
6551.12 6551.34 6552.24 6551.63 6552.28
Short Exit Signal for 428870 at 2024-07-23 17:34:40 at 2024-07-23 17:34:45.014716
6547.27 6547.26 6546.54 6547.09 6546.7
Long Entry Signal for 428870 at 2024-07-23 17:34:40 at 2024-07-23 17:34:45.018856
6547.27 6547.26 6546.54 6547.09 6546.7
Long Exit Signal for 428870 at 2024-07-23 17:35:05 at 2024-07-23 17:35:10.060920
6546.66 6547.08 6547.32 6547.32 6546.83
Long Entry Signal for 4288

In [31]:
resampled_data

{'428870':                        open    high     low   close  EMA_short  EMA_long  \
 tt                                                                         
 2024-07-23 17:19:25  6539.0  6539.0  6539.0  6539.0        NaN       NaN   
 2024-07-23 17:19:30  6541.0  6541.0  6541.0  6541.0        NaN       NaN   
 2024-07-23 17:19:45  6539.0  6539.0  6539.0  6539.0    6539.67       NaN   
 2024-07-23 17:20:15  6538.0  6538.0  6538.0  6538.0    6538.83       NaN   
 2024-07-23 17:20:35  6537.0  6538.0  6537.0  6538.0    6538.42       NaN   
 ...                     ...     ...     ...     ...        ...       ...   
 2024-07-23 17:38:10  6552.0  6552.0  6552.0  6552.0    6550.15   6547.62   
 2024-07-23 17:38:15  6554.0  6554.0  6554.0  6554.0    6552.07   6548.78   
 2024-07-23 17:38:25  6552.0  6552.0  6552.0  6552.0    6552.04   6549.37   
 2024-07-23 17:38:50  6553.0  6553.0  6553.0  6553.0    6552.52   6550.03   
 2024-07-23 17:38:55  6552.0  6552.0  6552.0  6552.0    6552.26   

In [ ]:
import pandas as pd
fno_scrips_mcx = pd.read_csv('//home/deep/Desktop/NEWshoonya/MCX_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
fno_scrips_mcx['StrikePrice'] = fno_scrips_mcx['StrikePrice'].astype(float)
fno_scrips_mcx.sort_values('Expiry',inplace=True)
fno_scrips_mcx.reset_index(drop=True, inplace=True)

fno_scrips = pd.read_csv('/home/deep/Desktop/NEWshoonya/NFO_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])
fno_scrips['StrikePrice'] = fno_scrips['StrikePrice'].astype(float)
fno_scrips.sort_values('Expiry',inplace=True)
fno_scrips.reset_index(drop=True, inplace=True)

In [ ]:
symbol = 'CRUDEOILM'
class TradingState:
    def __init__(self):
        self.ce_trading_symbol = None
        self.pe_trading_symbol = None
        self.ce_trading_token = None
        self.pe_trading_token = None

state = TradingState()

async def find_atm_strike_and_symbols():   
    global  atm_strike
    cmp_bn = float(resampled_data[ini]['close'].iloc[-1])      
    
    # In a real scenario, you might want to fetch this price asynchronously
    #cmp_bn = await get_current_price()  # You'd need to implement this function
    
    mod = int(cmp_bn) % 100
    cmp_bn = int(cmp_bn)
    atm_strike = cmp_bn - mod if mod <= 50 else cmp_bn + (100 - mod)
    #print(atm_strike)

    # Assuming fno_scrips_mcx is a global DataFrame
    filtered_df = fno_scrips_mcx[
        (fno_scrips_mcx['Symbol'] == symbol) &
        (fno_scrips_mcx['StrikePrice'] == float(atm_strike))
    ]
    
    if filtered_df.empty:
        print(f"Could not find trading symbols for ATM strike {atm_strike}")
        return None, None, None, None

    ce_row = filtered_df[filtered_df['OptionType'] == 'CE'].sort_values('Expiry').iloc[0]
    pe_row = filtered_df[filtered_df['OptionType'] == 'PE'].sort_values('Expiry').iloc[0]

    ce_trading_symbol, pe_trading_symbol = ce_row['TradingSymbol'], pe_row['TradingSymbol']
    ce_trading_token, pe_trading_token = ce_row['Token'], pe_row['Token']

    return ce_trading_symbol, pe_trading_symbol, ce_trading_token, pe_trading_token

async def update_atm_symbols():
    while True:
        try:
            result = await find_atm_strike_and_symbols()
            if result[0] is not None:
                (state.ce_trading_symbol, state.pe_trading_symbol, 
                 state.ce_trading_token, state.pe_trading_token) = result
                #print(f"Global symbols updated - CE: {state.ce_trading_symbol}, PE: {state.pe_trading_symbol}")
            else:
                print("Failed to update symbols")
        except Exception as e:
            print(f"Error in update_atm_symbols: {e}")
        await asyncio.sleep(15)

In [ ]:
state.ce_trading_symbol

In [32]:
def append_to_csv(trade_log):
    with open('trading_log.csv', 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow()

import pandas as pd

# Assuming resampled_data has the format {'token_name': DataFrame}
token, df = next(iter(resampled_data.items()))  # Get the token name and DataFrame

df.to_csv(f"{token}.csv", index=True)   # Save the DataFrame to a CSV file (include the index)
print(f"Saved data to {token}.csv")   


Saved data to 428870.csv


In [ ]:
op_chain = api.get_option_chain(exchange='NFO', tradingsymbol=state.ce_trading_symbol, strikeprice=atm_strike, count = 10)
tokens = [item['token'] for item in op_chain['values']]
subscriptions = [f"NFO|{token}" for token in tokens]
# Subscribe to each token
for sub in subscriptions:
    api.subscribe(sub)

# Print the subscriptions for verification
print("Subscribed to:", subscriptions)


In [3]:
import numpy as np
from numba import njit

@njit
def ema_numba(close, length):
    out = np.full_like(close, np.nan)
    
    if len(close) <= length:
        return out
    
    alpha = 2 / (length + 1)
    ema = np.mean(close[:length])  # Initialize with SMA
    out[length-1] = ema
    
    for i in range(length, len(close)):
        ema = alpha * close[i] + (1 - alpha) * ema
        out[i] = ema
    
    return np.round(out, 2)

import numpy as np
from numba import njit

@njit
def jma_numba(src, length, phase, power):
    out = np.full_like(src, np.nan)
    
    if len(src) <= length:
        return out
    
    phaseRatio = 1.5 + phase / 100 if -100 <= phase <= 100 else (0.5 if phase < -100 else 2.5)
    beta = 0.45 * (length - 1) / (0.45 * (length - 1) + 2)
    alpha = np.power(beta, power)
    
    e0 = e1 = e2 = jma = 0.0
    
    for i in range(len(src)):
        if i == 0:
            e0 = src[i]
            e1 = 0.0
            e2 = 0.0
            jma = 0.0
        else:
            e0 = (1 - alpha) * src[i] + alpha * e0
            e1 = (src[i] - e0) * (1 - beta) + beta * e1
            e2 = (e0 + phaseRatio * e1 - jma) * np.power(1 - alpha, 2) + np.power(alpha, 2) * e2
            jma = e2 + jma
            
            # Reset values if they become too large or unstable
            if np.abs(jma) > 1e10:
                e0 = e1 = e2 = jma = 0.0
        
        if i >= length - 1:
            out[i] = np.round(jma, 2)
    
    return out

import numpy as np
from numba import njit

@njit
def alma_numba(series, length, offset, sigma):
    out = np.full_like(series, np.nan)
    
    if len(series) < length:
        return out
    
    for i in range(length - 1, len(series)):
        numerator = 0.0
        denominator = 0.0
        m = offset * (length - 1)
        s = length / sigma
        for j in range(length):
            weight = np.exp(-((j - m) ** 2) / (2 * s * s))
            numerator += weight * series[i - length + 1 + j]
            denominator += weight
        out[i] = round(numerator / denominator, 2)
    
    return out

# Usage:
# resampled_data[token]['EMA_short'] = ema_numba(resampled_data[token]['close'].values, length=3)
# resampled_data[token]['EMA_long'] = ema_numba(resampled_data[token]['close'].values, length=10)
# resampled_data[token]['JMA'] = jma_numba(resampled_data[token]['close'].values, length=7, phase=50, power=2)

long
Latest price for 48741 is 493.3
Long Entry Signal for 26009 at 2024-07-23 12:21:45 at 2024-07-23 12:22:13.837421
Latest price for 48741 is 470.5
Long Exit Signal for 26009 at 2024-07-23 12:22:30 at 2024-07-23 12:22:56.490288
long
Latest price for 48735 is 486.35
Long Entry Signal for 26009 at 2024-07-23 12:25:30 at 2024-07-23 12:25:46.130108
Latest price for 48735 is 434.9
Long Exit Signal for 26009 at 2024-07-23 12:27:15 at 2024-07-23 12:27:30.002809
long
Latest price for 48731 is 387.5
Long Entry Signal for 26009 at 2024-07-23 12:32:45 at 2024-07-23 12:33:00.001236
Latest price for 48731 is 494.65
Long Exit Signal for 26009 at 2024-07-23 12:36:45 at 2024-07-23 12:37:32.334099
SHORT
Latest price for 48740 is 362.6
Short Entry Signal for 26009 at 2024-07-23 12:36:45 at 2024-07-23 12:37:32.334561
Latest price for 48740 is 340.0
Short Exit Signal for 26009 at 2024-07-23 12:38:15 at 2024-07-23 12:39:10.906132
long
Latest price for 48741 is 363.5
Long Entry Signal for 26009 at 2024-07-23 12:38:15 at 2024-07-23 12:39:10.913781
Latest price for 48741 is 339.4
Long Exit Signal for 26009 at 2024-07-23 12:39:15 at 2024-07-23 12:40:14.871653
long
Latest price for 48741 is 348.6
Long Entry Signal for 26009 at 2024-07-23 12:41:15 at 2024-07-23 12:42:23.841913
Latest price for 48741 is 347.6
Long Exit Signal for 26009 at 2024-07-23 12:44:15 at 2024-07-23 12:45:32.840069
SHORT
Latest price for 48744 is 350.85
Short Entry Signal for 26009 at 2024-07-23 12:44:15 at 2024-07-23 12:45:32.840540
Latest price for 48744 is 371.6
Short Exit Signal for 26009 at 2024-07-23 12:50:30 at 2024-07-23 12:51:54.132132
long
Latest price for 48735 is 331.2
Long Entry Signal for 26009 at 2024-07-23 12:50:30 at 2024-07-23 12:51:54.133512
Latest price for 48735 is 302.1
Long Exit Signal for 26009 at 2024-07-23 12:52:15 at 2024-07-23 12:53:38.992529
SHORT
Latest price for 48740 is 315.15
Short Entry Signal for 26009 at 2024-07-23 12:52:15 at 2024-07-23 12:53:38.992773
Latest price for 48740 is 289.5
Short Exit Signal for 26009 at 2024-07-23 12:53:15 at 2024-07-23 12:54:40.068937
SHORT
Latest price for 48744 is 337.25
Short Entry Signal for 26009 at 2024-07-23 12:55:30 at 2024-07-23 12:56:55.059689
Latest price for 48744 is 424.45
Short Exit Signal for 26009 at 2024-07-23 13:01:45 at 2024-07-23 13:02:00.354596
long
Latest price for 48733 is 307.65
Long Entry Signal for 26009 at 2024-07-23 13:01:45 at 2024-07-23 13:02:00.356891
Latest price for 48733 is 293.45
Long Exit Signal for 26009 at 2024-07-23 13:02:15 at 2024-07-23 13:02:30.006587
long
Latest price for 48733 is 289.65
Long Entry Signal for 26009 at 2024-07-23 13:02:30 at 2024-07-23 13:02:45.004242
Latest price for 48733 is 285.8
Long Exit Signal for 26009 at 2024-07-23 13:03:30 at 2024-07-23 13:03:45.004909
SHORT
Latest price for 48734 is 336.4
Short Entry Signal for 26009 at 2024-07-23 13:03:30 at 2024-07-23 13:03:45.006044
Latest price for 48734 is 326.9
Short Exit Signal for 26009 at 2024-07-23 13:06:00 at 2024-07-23 13:06:15.003464
long
Latest price for 48733 is 284.55
Long Entry Signal for 26009 at 2024-07-23 13:06:00 at 2024-07-23 13:06:15.004283
Latest price for 48733 is 267.8
Long Exit Signal for 26009 at 2024-07-23 13:07:00 at 2024-07-23 13:07:15.003902
SHORT
Latest price for 48732 is 292.25
Short Entry Signal for 26009 at 2024-07-23 13:07:00 at 2024-07-23 13:07:15.004560
Latest price for 48732 is 402.4
Short Exit Signal for 26009 at 2024-07-23 13:15:45 at 2024-07-23 13:16:00.003695
long
Latest price for 48727 is 334.6
Long Entry Signal for 26009 at 2024-07-23 13:15:45 at 2024-07-23 13:16:00.005043
Latest price for 48727 is 314.9
Long Exit Signal for 26009 at 2024-07-23 13:16:30 at 2024-07-23 13:16:45.004542
SHORT
Latest price for 48728 is 313.1
Short Entry Signal for 26009 at 2024-07-23 13:18:15 at 2024-07-23 13:18:30.014488
Latest price for 48728 is 283.7
Short Exit Signal for 26009 at 2024-07-23 13:19:15 at 2024-07-23 13:19:30.007336
long
Latest price for 48727 is 332.95
Long Entry Signal for 26009 at 2024-07-23 13:19:15 at 2024-07-23 13:19:30.008724
Latest price for 48727 is 320.3
Long Exit Signal for 26009 at 2024-07-23 13:22:00 at 2024-07-23 13:22:15.003740
SHORT
Latest price for 48728 is 283.15
Short Entry Signal for 26009 at 2024-07-23 13:22:00 at 2024-07-23 13:22:15.004514
Latest price for 48728 is 260.8
Short Exit Signal for 26009 at 2024-07-23 13:24:00 at 2024-07-23 13:24:15.006135
long
Latest price for 48729 is 282.55
Long Entry Signal for 26009 at 2024-07-23 13:24:00 at 2024-07-23 13:24:15.007297
Latest price for 48729 is 312.55
Long Exit Signal for 26009 at 2024-07-23 13:32:45 at 2024-07-23 13:33:00.004567
SHORT
Latest price for 48732 is 289.95
Short Entry Signal for 26009 at 2024-07-23 13:32:45 at 2024-07-23 13:33:00.005397
Latest price for 48732 is 276.15
Short Exit Signal for 26009 at 2024-07-23 13:36:30 at 2024-07-23 13:36:45.002587
long
Latest price for 48731 is 270.0
Long Entry Signal for 26009 at 2024-07-23 13:36:30 at 2024-07-23 13:36:45.002920
Latest price for 48731 is 290.0
Long Exit Signal for 26009 at 2024-07-23 13:40:45 at 2024-07-23 13:41:00.001750
SHORT
Latest price for 48732 is 245.65
Short Entry Signal for 26009 at 2024-07-23 13:40:45 at 2024-07-23 13:41:00.002056
Latest price for 48732 is 244.0
Short Exit Signal for 26009 at 2024-07-23 13:41:00 at 2024-07-23 13:41:15.002399
SHORT
Latest price for 48732 is 262.9
Short Entry Signal for 26009 at 2024-07-23 13:41:45 at 2024-07-23 13:42:00.004119
Latest price for 48732 is 252.35
Short Exit Signal for 26009 at 2024-07-23 13:42:45 at 2024-07-23 13:43:00.002614
SHORT
Latest price for 48732 is 256.8
Short Entry Signal for 26009 at 2024-07-23 13:44:45 at 2024-07-23 13:45:00.002247
Latest price for 48732 is 259.5
Short Exit Signal for 26009 at 2024-07-23 13:47:30 at 2024-07-23 13:47:45.002487
long
Latest price for 48731 is 280.35
Long Entry Signal for 26009 at 2024-07-23 13:47:30 at 2024-07-23 13:47:45.010203
Latest price for 48731 is 269.2
Long Exit Signal for 26009 at 2024-07-23 13:51:15 at 2024-07-23 13:51:30.004648
SHORT
Latest price for 48732 is 270.8
Short Entry Signal for 26009 at 2024-07-23 13:51:15 at 2024-07-23 13:51:30.005676
Latest price for 48732 is 254.8
Short Exit Signal for 26009 at 2024-07-23 13:53:00 at 2024-07-23 13:53:15.002012
long
Latest price for 48733 is 228.4
Long Entry Signal for 26009 at 2024-07-23 13:53:00 at 2024-07-23 13:53:15.002362